# Explainable Credit PD Model + SR 11-7 Validation Pack

**Two modules, one portfolio piece.**

1. **Lending Club -> Probability-of-Default (PD) model** — real loan outcomes, no protected attributes. Carries PD modeling, SHAP explainability, and the validation report.
2. **HMDA -> Fair-lending audit** — real applicant demographics + approve/deny decision, no default outcome. Carries the disparate-impact analysis.

Done from two seats: **first line (developer)** builds the model; **second line (validator)** reviews it and issues findings. Headline artifact: [`reports/validation_report.md`](../reports/validation_report.md).

> **Data note.** Real `data/lending_club.csv` / `data/hmda_2024.csv` are used if present; otherwise realistic **synthetic** data is generated and all numbers are illustrative of method, not a real portfolio.

## 0. Setup

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))  # import src... from notebooks/
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split

import src.data as data, src.scorecard as scorecard, src.challenger as challenger
import src.metrics as metrics, src.explain as explain, src.fairness as fairness

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True})
F = data.LC_UNDERWRITING_FEATURES

# MODULE 1 — Lending Club PD model
## 1. Data: target definition, leakage audit, out-of-time split

In [2]:
raw, lc_is_real = data.load_lending_club()
print('Real Lending Club data:', lc_is_real, '| raw rows:', len(raw))
mdf, meta = data.define_target_and_audit(raw)
audit = meta['audit']
print('rows kept:', audit['rows_kept'], '| default rate:', audit['default_rate'])
print('dropped unresolved:', audit['rows_dropped_unresolved'])
print('leakage dropped:', audit['leakage_columns_dropped'])
print('lender-assessment excluded:', audit['lender_assessment_excluded'])

Real Lending Club data: False | raw rows: 60000
rows kept: 56943 | default rate: 0.2244
dropped unresolved: 3057
leakage dropped: ['recoveries', 'collection_recovery_fee', 'total_pymnt', 'total_rec_prncp', 'total_rec_int', 'last_pymnt_amnt', 'out_prncp', 'funded_amnt']
lender-assessment excluded: ['grade', 'sub_grade', 'int_rate']


**Leakage demonstration** — including post-origination fields makes default trivially 'predictable'. That near-1.0 AUC is the signature of leakage.

In [3]:
leak_df, leak_cols = meta['leak_demo_df'], meta['leak_cols']
ltr = leak_df[leak_df.issue_year.isin([2014,2015,2016])]
loot = leak_df[leak_df.issue_year == 2018]
g1 = HistGradientBoostingClassifier(max_depth=4, max_iter=150, random_state=0).fit(ltr[data.LC_NUMERIC+leak_cols], ltr.default)
g2 = HistGradientBoostingClassifier(max_depth=4, max_iter=150, random_state=0).fit(ltr[data.LC_NUMERIC], ltr.default)
print('OOT AUC WITH leakage:', round(metrics.roc_auc_score(loot.default, g1.predict_proba(loot[data.LC_NUMERIC+leak_cols])[:,1]),4), ' <- red flag')
print('OOT AUC clean       :', round(metrics.roc_auc_score(loot.default, g2.predict_proba(loot[data.LC_NUMERIC])[:,1]),4))

OOT AUC WITH leakage: 1.0  <- red flag
OOT AUC clean       : 0.7592


In [4]:
tr, va, oot = data.oot_split(mdf)
print('train/valid/oot:', len(tr), len(va), len(oot))
print('default rates:', round(tr.default.mean(),3), round(va.default.mean(),3), round(oot.default.mean(),3))

train/valid/oot: 35987 12064 8892
default rates: 0.217 0.238 0.238


## 2. Champion — WOE scorecard
Bin with Weight of Evidence, rank by Information Value, fit logistic regression on WOE, scale to points.

In [5]:
enc = scorecard.WOEEncoder(data.LC_NUMERIC, data.LC_CATEGORICAL).fit(tr[F], tr.default)
print('IV ranking:'); print(enc.iv_table().round(4))

IV ranking:
                    IV strength
fico            0.3187   strong
revol_util      0.2692   medium
dti             0.1026   medium
inq_last_6mths  0.0318     weak
annual_inc      0.0249     weak
home_ownership  0.0099  useless
purpose         0.0098  useless
emp_length      0.0054  useless
open_acc        0.0006  useless
loan_amnt       0.0003  useless
term_months     0.0000  useless


In [6]:
sc = scorecard.Scorecard(enc, base_score=600, base_odds=50, pdo=20).fit(tr[F], tr.default)
print('features kept (IV >= 0.02):', sc.features_)
print('dropped low-IV:', sc.dropped_low_iv_)
print('\nWald coefficients (manual (XtVX)^-1):'); print(sc.wald_.round(4))

features kept (IV >= 0.02): ['fico', 'dti', 'annual_inc', 'revol_util', 'inq_last_6mths']
dropped low-IV: ['loan_amnt', 'term_months', 'emp_length', 'open_acc', 'purpose', 'home_ownership']

Wald coefficients (manual (XtVX)^-1):
                  coef  std_err        z  p_value
intercept      -1.2880   0.0141 -91.3575      0.0
fico           -1.0883   0.0253 -42.9505      0.0
dti            -1.1041   0.0436 -25.3400      0.0
annual_inc     -1.1445   0.0892 -12.8302      0.0
revol_util     -1.1076   0.0267 -41.4739      0.0
inq_last_6mths -1.1738   0.0750 -15.6523      0.0


In [7]:
p_ch = sc.predict_proba(oot[F])
print('Champion OOT discrimination:', metrics.discrimination(oot.default, p_ch))

Champion OOT discrimination: {'AUC': 0.7481, 'Gini': 0.4961, 'KS': 0.3677}


## 3. Challenger — monotone-constrained, calibrated gradient boosting
Hard monotone constraints encode economic priors; isotonic calibration keeps PD *levels* usable for pricing/ECL.

In [8]:
clf, feats, mono = challenger.build_challenger(data.LC_NUMERIC, data.LC_CATEGORICAL, data.LC_MONOTONE, calibrate=True)
clf.fit(tr[F], tr.default)
p_cl = clf.predict_proba(oot[F])[:,1]
print('Challenger OOT discrimination:', metrics.discrimination(oot.default, p_cl))
monocheck = {f: challenger.check_monotonicity(clf, tr[F], f, data.LC_MONOTONE[f]) for f in data.LC_NUMERIC if data.LC_MONOTONE[f] != 0}
print('monotonicity verified:', monocheck)

Challenger OOT discrimination: {'AUC': 0.7658, 'Gini': 0.5316, 'KS': 0.4001}


monotonicity verified: {'fico': True, 'dti': True, 'annual_inc': True, 'loan_amnt': True, 'term_months': True, 'emp_length': True, 'revol_util': True, 'inq_last_6mths': True}


## 4. Outcomes analysis — discrimination, calibration, business view
Report all three metric families; calibration is where the champion fails (Hosmer-Lemeshow p < 0.05).

In [9]:
print('Champion  calibration:', metrics.calibration(oot.default, p_ch))
print('Challenger calibration:', metrics.calibration(oot.default, p_cl))

Champion  calibration: {'Brier': 0.15492, 'HL_chi2': 25.13, 'dof': 8, 'p_value': 0.0015}
Challenger calibration: {'Brier': 0.14932, 'HL_chi2': 18.58, 'dof': 8, 'p_value': 0.0173}


In [10]:
ead = oot['loan_amnt'].values
el_ch = metrics.expected_loss(p_ch, ead); el_cl = metrics.expected_loss(p_cl, ead)
print('Expected loss champion :', round(el_ch))
print('Expected loss challenger:', round(el_cl))
print('reduction from challenger:', round(el_ch - el_cl), f'({100*(el_ch-el_cl)/el_ch:.1f}%)')

Expected loss champion : 19097897
Expected loss challenger: 18039633
reduction from challenger: 1058264 (5.5%)


## 5. Stability — PSI train -> OOT
PSI is *the* monitoring metric in credit. Watch for input drift preceding output drift.

In [11]:
for col in ['dti','fico']:
    r = metrics.psi_report(tr[col], oot[col]); print(f'{col:>5} -> PSI {r["PSI"]:.4f} | {r["band"]}')
sp = metrics.psi_report(clf.predict_proba(tr[F])[:,1], p_cl)
print(f'score -> PSI {sp["PSI"]:.4f} | {sp["band"]}')

  dti -> PSI 0.1148 | moderate shift
 fico -> PSI 0.0553 | stable
score -> PSI 0.0290 | stable


## 6. Explainability — SHAP, reason codes, conceptual-soundness cross-check

In [12]:
bg = tr[F].sample(120, random_state=1)
predict = lambda d: clf.predict_proba(d)[:,1]
samp = oot[F].sample(180, random_state=2)
sm = explain.shap_matrix(predict, samp, bg, F, nsamples=120, seed=3)
print('SHAP global importance (mean |phi|):'); print(explain.global_importance(sm).round(4))

SHAP global importance (mean |phi|):
fico              0.1049
revol_util        0.1016
dti               0.0682
inq_last_6mths    0.0608
annual_inc        0.0571
term_months       0.0522
home_ownership    0.0520
emp_length        0.0486
open_acc          0.0485
purpose           0.0466
loan_amnt         0.0452
dtype: float64


In [13]:
# adverse-action reason codes for the highest-PD (declined) applicant
hi = oot[F].iloc[[np.argmax(p_cl)]]
phi = explain.kernel_shap(predict, hi.iloc[0], bg, F, nsamples=500, seed=9)
print('SHAP reason codes     :', explain.reason_codes_from_shap(phi, 4))
print('Scorecard reason codes:', explain.reason_codes_from_scorecard(sc, hi.iloc[0], 4))

SHAP reason codes     : [('revol_util', 0.2742, 'Revolving credit utilisation too high'), ('fico', 0.2357, 'Credit score below approved-applicant norms'), ('inq_last_6mths', 0.1151, 'Too many recent credit inquiries'), ('purpose', 0.0728, 'Stated loan purpose carries higher risk')]
Scorecard reason codes: [('revol_util', -19.3, 'Revolving credit utilisation too high'), ('fico', -17.6, 'Credit score below approved-applicant norms'), ('dti', -8.0, 'Debt-to-income ratio too high'), ('inq_last_6mths', -6.0, 'Too many recent credit inquiries')]


In [14]:
cs = explain.conceptual_soundness(sm, samp, sc.predict_proba(samp), data.LC_MONOTONE, data.LC_NUMERIC)
print('Sign agreement (strong features should all agree; weak ones may flip):'); print(cs)

Sign agreement (strong features should all agree; weak ones may flip):
          feature  challenger_sign  champion_sign  theory_sign  all_agree
0            fico               -1             -1           -1       True
1             dti                1              1            1       True
2      annual_inc               -1              1           -1      False
3       loan_amnt                1             -1            1      False
4     term_months                1             -1            1      False
5      emp_length               -1             -1           -1       True
6      revol_util                1              1            1       True
7  inq_last_6mths                1              1            1       True
8        open_acc               -1             -1            0       True


## 7. Implementation testing
An independent re-fit must reproduce the reported result.

In [15]:
clf2, _, _ = challenger.build_challenger(data.LC_NUMERIC, data.LC_CATEGORICAL, data.LC_MONOTONE, calibrate=True)
clf2.fit(tr[F], tr.default)
auc2 = metrics.roc_auc_score(oot.default, clf2.predict_proba(oot[F])[:,1])
reported = metrics.discrimination(oot.default, p_cl)['AUC']
print('rerun OOT AUC:', round(auc2,4), '| reported:', reported, '| reproducible:', round(auc2,4)==reported)

rerun OOT AUC: 0.7658 | reported: 0.7658 | reproducible: True


# MODULE 2 — HMDA fair-lending audit
Separate data, separate competency: auditing a real lending **decision** for disparate impact. No PD model is trained here (HMDA has no default outcome).

In [16]:
h, hmda_is_real = data.load_hmda(); h = data.hmda_prepare(h)
print('Real HMDA data:', hmda_is_real, '| rows:', len(h), '| denial rate:', round(h.denied.mean(),4))

Real HMDA data: False | rows: 40000 | denial rate: 0.6027


## 8. Observed disparate impact (the decisions themselves)
Adverse-impact ratio = group approval rate / best-group rate. Below 0.80 fails the four-fifths rule.

In [17]:
air_obs = fairness.adverse_impact_ratio(h.derived_race, 1 - h.denied)
print('reference group:', air_obs.attrs['reference_group'])
print(air_obs[['approval_rate','n','AIR_vs_ref','flag_80pct_rule']].round(3))

reference group: Joint
                                  approval_rate      n  AIR_vs_ref  \
g                                                                    
Joint                                     0.429   2409       1.000   
White                                     0.427  24857       0.994   
Asian                                     0.420   5133       0.979   
American Indian or Alaska Native          0.305   1184       0.710   
Black or African American                 0.270   6417       0.628   

                                  flag_80pct_rule  
g                                                  
Joint                                       False  
White                                       False  
Asian                                       False  
American Indian or Alaska Native             True  
Black or African American                    True  


## 9. Does a model on *legitimate features only* remove the disparity? (No.)
Excluding protected attributes does not remove disparate impact when the legitimate features are correlated with group membership — 'fairness through unawareness' fails.

In [18]:
X, y, grp = h[data.HMDA_FEATURES], h.denied, h.derived_race
Xtr, Xte, ytr, yte, gtr, gte = train_test_split(X, y, grp, test_size=0.3, random_state=0, stratify=y)
base = HistGradientBoostingClassifier(max_depth=4, max_iter=200, random_state=0).fit(Xtr, ytr)
p = base.predict_proba(Xte)[:,1]; appr = (p < 0.5).astype(int)
res = fairness.summarize(gte, yte, p, appr)
print('baseline AUC:', res['auc'], '| min AIR:', res['min_AIR'])
print('failing 80% rule:', res['groups_failing_80pct'])
print('parity gaps:', res['parity_gaps'])

baseline AUC: 0.6801 | min AIR: 0.333
failing 80% rule: ['American Indian or Alaska Native', 'Black or African American']
parity gaps: {'TPR_equal_opportunity': 0.129, 'precision_predictive_parity': 0.115, 'FPR': 0.248}


## 10. Mitigation and its cost
Reweighing is fairness-safe but weak; group thresholds work but are a disparate-*treatment* concern. The effective lever is the legally fraught one.

In [19]:
# mitigation 1: Kamiran-Calders reweighing
w = fairness.reweighing_weights(gtr, ytr)
mit = HistGradientBoostingClassifier(max_depth=4, max_iter=200, random_state=0).fit(Xtr, ytr, sample_weight=w)
pm = mit.predict_proba(Xte)[:,1]; apprm = (pm < 0.5).astype(int)
resm = fairness.summarize(gte, yte, pm, apprm)
print('reweighed -> AUC:', resm['auc'], '| min AIR:', resm['min_AIR'], '(little movement)')

reweighed -> AUC: 0.6788 | min AIR: 0.337 (little movement)


In [20]:
# mitigation 2: per-group thresholds to reach the 80% rule
appr_thr, thr = fairness.group_thresholds_for_air(gte, p, target_air=0.80)
air_thr = fairness.adverse_impact_ratio(pd.Series(gte.values), pd.Series(appr_thr))
print('group thresholds:', {k: round(v,3) for k,v in thr.items()})
print('min AIR after thresholds:', round(float(air_thr['AIR_vs_ref'].min()),3), '(works, but disparate-treatment risk)')

group thresholds: {'White': 0.5, 'Asian': 0.5, 'Joint': 0.5, 'Black or African American': 0.596, 'American Indian or Alaska Native': 0.577}
min AIR after thresholds: 0.801 (works, but disparate-treatment risk)


---
## Reproduce everything
`python ../src/run_pipeline.py` regenerates `reports/results.json` and all 11 figures used in the validation report.

**Headline deliverable:** [`reports/validation_report.md`](../reports/validation_report.md).